In [2]:
import anthropic
from anthropic.types import Message
from datetime import datetime

model = "claude-sonnet-4-6"
client = anthropic.Anthropic()

In [3]:
def get_current_datetime(date_format="%Y-%m-%d %H:%M:%S"):
    if not date_format:
        raise ValueError("date_format cannot be empty")
    return datetime.now().strftime(date_format)

get_current_datetime_schema = {
    "name": "get_current_datetime",
    "description": "Returns the current date and time formatted according to the specified format. Use this when you need to know the current date or time. Returns a string with the formatted datetime.",
    "input_schema": {
        "type": "object",
        "properties": {
            "date_format": {
                "type": "string",
                "description": "A string specifying the format of the returned datetime. Uses Python's strftime format codes.",
                "default": "%Y-%m-%d %H:%M:%S"
            }
        },
        "required": []
    }
}

In [4]:
def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message
    }
    messages.append(user_message)

def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message
    }
    messages.append(assistant_message)

def text_from_message(message):
    return "\n".join(
        [block.text for block in message.content if block.type == "text"]
    )

def chat(messages, system=None, temperature=1.0, stop_sequences=[], tools=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }
    if tools:
        params["tools"] = tools
    if system:
        params["system"] = system
    return client.messages.create(**params)

In [5]:
def run_tools(response):
    tool_result_blocks = []
    for block in response.content:
        if block.type == "tool_use":
            if block.name == "get_current_datetime":
                result = get_current_datetime(**block.input)
            tool_result_blocks.append({
                "type": "tool_result",
                "tool_use_id": block.id,
                "content": str(result),
                "is_error": False
            })
    return tool_result_blocks

def run_conversation(messages, tools):
    while True:
        response = chat(messages, tools=tools)
        add_assistant_message(messages, response)

        if response.stop_reason != "tool_use":
            break

        tool_result_blocks = run_tools(response)
        add_user_message(messages, tool_result_blocks)

    return messages

In [7]:
messages = []
add_user_message(messages, "What is the exact time, formatted as HH:MM:SS?")
messages = run_conversation(messages, tools=[get_current_datetime_schema])
last_response = messages[-1]["content"]
print("\n".join([block.text for block in last_response if block.type == "text"]))

The exact current time is **14:31:08** (in HH:MM:SS format).


# Testing the Conversation Loop

We test our `run_conversation` function by asking Claude for the current time.

The conversation flow works like this:
1. We add the user message to an empty messages list
2. `run_conversation` sends it to Claude with the tool schema
3. Claude requests `get_current_datetime`, our code runs it and sends the result back
4. Claude returns the final answer

For printing the result, we access `messages[-1]["content"]` which is the last assistant message. Since it contains a list of blocks, we loop through and grab only the text blocks to display.